<a href="https://colab.research.google.com/github/adikatre/Asymmetric-Cross-Modal-Attention/blob/main/notebooks/02_train_evaluate_visualize_colab_unfrozen_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

note on some ideas to consider:

Rigorous Critical Evaluation (read this before writing the paper)
You asked for rigorous evaluation. ICDM reviewers will find these issues — better to address them now.

Single seed (SEED=42). Your code uses one seed. ICDM expects ≥3 seeds with mean ± std and a paired test. Differences below
~0.7 VQA-acc points are typically within run noise. Run at least 3 seeds for both models before claiming superiority.
Parameter-count confound. AsymmetricCrossModalFusion has two full cross-attention blocks

(~2× the fusion parameters of the symmetric baseline). A reviewer will say: "Your win is just from more parameters." You must add a parameter-matched baseline:

Option A: Symmetric model with 2 stacked shared blocks (same param count, no asymmetry)
Option B: Symmetric block with 2× the FFN expansion ratio


Confounded mechanism. Your asymmetric model differs from the baseline along two axes simultaneously: (a) independent weights per direction, (b) sequential information flow (text attends to grounded image, not raw image). You cannot attribute the gain to "asymmetry" without a 2×2 factorial:
ParallelSequentialShared weightsSymmetric (your baseline)new ablationIndependent weightsnew ablationAsymmetric (your model)

3 epochs is severely undertrained for VQA. Standard VQA papers train 13–25 epochs. If both models are still improving, your comparison at epoch 3 measures learning speed, not converged performance. Plot val accuracy vs. epoch and verify both have plateaued — or run longer.
Top-1000 answer vocabulary caps maximum achievable VQA accuracy at ~88% (vs. ~93% with the standard top-3000). State this explicitly and justify; reviewers will compare to numbers from papers using top-3000.
No comparison to published VQA baselines. Even a citation table — BUTD (Anderson et al., 2018), MCAN (Yu et al., 2019), ViLBERT, LXMERT — is required to position the work. Without it, "asymmetric beats symmetric" reads as a pedagogical exercise, not a contribution.

Issues that weaken the paper

Sequential ordering is arbitrary. You go image → text. Why not text → image? Or two iterations? An ablation over orderings strengthens the claim that "first-grounding-then-querying" is what helps.
Attention claims are anecdotal. "More focused attention" needs a number — attention entropy, or IoU with VQA-HAT (Das et al., 2017) human attention bounding boxes — not just heatmap pictures.
Modality ablation interpretation is dangerous. If your asymmetric model achieves higher Image-Blind accuracy, that may indicate it's more language-biased, which is a known VQA failure mode (cf. VQA-CP). Frame this carefully or it backfires.
Test-dev numbers from the official VQA evaluation server are not reported. You generate the JSON in cell 77 — submit it. A leaderboard number is far more credible than self-evaluated val accuracy.
Soft-target masking biases the dataset. Cell 21 silently drops questions where no annotator answer is in the top-1000 vocabulary. This makes both models look better and inflates accuracy. Report sample counts before/after filtering and per-question-type retention rates.

## Load and Unzip VQA Dataset

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/sample_data/

Mounted at /content/drive


In [ ]:
# Get vqa data from drive opt.
!cp "/content/drive/MyDrive/VQA.zip" "/content/"

In [2]:
# make data dirs
!mkdir -p /content/data/
!mkdir -p /content/data/answers
!mkdir -p /content/data/images
!mkdir -p /content/data/questions

#copy over zips (https://drive.google.com/drive/folders/1VJ1xNxo_dAGJ4ZcpaFQBo-wpdIChZkpx?usp=sharing) from drive into here
!cp -r /content/drive/MyDrive/VQA/ /content/data/zip/

In [ ]:
# unqip vqa.zip, then remove it opt.
!unzip -q /content/VQA.zip -d /content/data/zip

!rm -rf /content/VQA.zip

In [ ]:
# test2015 opt.
!unzip -q /content/data/zip/test2015.zip -d /content/data/images/

!rm -rf /content/data/zip/test2015.zip

In [ ]:
# train2014 opt.
!unzip -q /content/data/zip/train2014.zip -d /content/data/images/

!rm -rf /content/data/zip/train2014.zip

In [ ]:
# val2014 opt.
!unzip -q /content/data/zip/val2014.zip -d /content/data/images/

!rm -rf /content/data/zip/val2014.zip

In [3]:
# extract annotations (labels)
!unzip -q /content/data/zip/v2_Annotations_Train_mscoco.zip -d /content/data/answers/
!unzip -q /content/data/zip/v2_Annotations_Val_mscoco.zip -d /content/data/answers/

!rm -rf /content/data/zip/v2_Annotations_Train_mscoco.zip
!rm -rf /content/data/zip/v2_Annotations_Val_mscoco.zip

In [4]:
# extract testing data
!unzip -q /content/data/zip/v2_Questions_Test_mscoco.zip -d /content/data/questions/
!unzip -q /content/data/zip/v2_Questions_Train_mscoco.zip -d /content/data/questions/
!unzip -q /content/data/zip/v2_Questions_Val_mscoco.zip -d /content/data/questions/

!rm -rf /content/data/zip/v2_Questions_Test_mscoco.zip
!rm -rf /content/data/zip/v2_Questions_Train_mscoco.zip
!rm -rf /content/data/zip/v2_Questions_Val_mscoco.zip

In [ ]:
# copy over past results for resume function
!mkdir -p /content/results
!cp -r /content/drive/MyDrive/unfrozen_results_4_25/final_results/* /content/results/

In [5]:
# copy over vqa images h5 from VQA dir
!cp /content/drive/MyDrive/VQA_cache/vqa_images_336.h5 /content/data/


# 2 — Train, Evaluate & Visualize

Complete experiment in one notebook:

1. **Data** — Load VQA v2.0 dataset
2. **Models** — Define encoders, attention blocks, and full VQA models
3. **Training** — Train symmetric baseline and asymmetric model
4. **Evaluation** — Compare metrics (Top-1, Top-5 accuracy)
5. **Visualization** — Training curves, attention heatmaps, qualitative examples

**Run all cells in order.** Edit the configuration cell below to change hyperparameters.

In [6]:
!pip install -q torch torchvision transformers sentencepiece timm matplotlib tqdm Pillow h5py bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.9 MB/s eta 0:00:00


In [7]:
import json
import random
import time
from collections import Counter
from contextlib import nullcontext
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer
import bitsandbytes as bnb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


## Configuration

Edit these variables to change the experiment.

In [8]:
# Paths
DATA_DIR       = Path("/content/data")
CHECKPOINT_DIR = Path("/content/results/checkpoints")
METRICS_DIR    = Path("/content/results/metrics")
FIGURES_DIR    = Path("/content/results/figures")

# Model
NUM_ANSWERS      = 3129    # was 1000 -- standard VQAv2 vocab (Pythia/MCAN), covers ~93% of train
EMBED_DIM        = 1024
NUM_HEADS        = 16
FUSION_DEPTH     = 6       # number of (self-attn + cross-attn) fusion layers
DROPOUT          = 0.1     # FFN/projection dropout across model
ATTN_DROPOUT     = 0.1     # multi-head attention softmax dropout
CLS_DROPOUT      = 0.3     # final classifier head dropout
FREEZE_ENCODERS  = False   # True = offline features; False = end-to-end training

# Data
MAX_QUESTION_LEN = 20
MAX_SAMPLES      = None    # set to 1000 for a quick dev run, and None for a complete run

# Training
if FREEZE_ENCODERS:
    BATCH_SIZE    = 128    # precomputed features are small
    LEARNING_RATE = 1e-4
else:
    BATCH_SIZE    = 16     # L4 24GB: BS=16 + GRAD_ACCUM=6 keeps effective batch at 96
    LEARNING_RATE = 1e-4   # fusion + classifier LR
    ENCODER_LR    = 1e-5   # top-layer encoder LR (LLRD decays earlier layers further)
    WARMUP_EPOCHS = 2      # linear warmup to avoid catastrophic forgetting

LLRD_DECAY       = 0.95    # layer-wise LR decay factor for the per-layer encoder groups
GRAD_ACCUM_STEPS = 6       # micro-batches per optimizer step (effective BS = BATCH_SIZE * 6 = 96)
WEIGHT_DECAY     = 1e-2    # was 1e-5 -- standard AdamW for transformer fine-tunes (LN/bias excluded)
EPOCHS           = 18      # was 13 -- cosine schedule needs room to decay
NUM_WORKERS      = 4
SEED             = 42
USE_AMP          = True
AMP_DTYPE        = torch.bfloat16  # bf16 has fp32 dynamic range => no GradScaler needed

for d in [CHECKPOINT_DIR, METRICS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)


---
## 1. Data Loading

In [9]:
import re

# VQAv2 official-style answer normalization (lowercase, strip punctuation,
# remove articles, fix common contractions). Applied identically when building
# the vocabulary and when constructing soft targets so they stay in sync.
_ARTICLES = {"a", "an", "the"}
_PUNCT_RE = re.compile(r"[^\w\s\']")
_CONTRACTIONS = {
    "dont": "don't", "doesnt": "doesn't", "didnt": "didn't",
    "isnt": "isn't", "arent": "aren't",
    "wasnt": "wasn't", "werent": "weren't",
    "wont": "won't", "cant": "can't", "couldnt": "couldn't",
    "wouldnt": "wouldn't", "shouldnt": "shouldn't",
    "havent": "haven't", "hasnt": "hasn't", "hadnt": "hadn't",
    "thats": "that's", "whats": "what's", "wheres": "where's",
    "theres": "there's", "heres": "here's", "youre": "you're",
    "theyre": "they're", "weve": "we've", "youve": "you've",
}


def normalize_answer(s: str) -> str:
    """Lowercase, strip punctuation, drop articles, fix contractions."""
    s = s.lower().strip()
    s = _PUNCT_RE.sub(" ", s)
    s = " ".join(t for t in s.split() if t not in _ARTICLES)
    return _CONTRACTIONS.get(s, s)


def build_answer_vocab(annotations_file, top_k=3129):
    """Build answer vocabulary from the top_k most frequent normalized answers.

    Counts every annotator answer (10 per question), not just the
    `multiple_choice_answer`, which gives a more representative distribution.
    """
    with open(annotations_file) as f:
        annotations = json.load(f)["annotations"]

    counter = Counter()
    for ann in annotations:
        for a in ann["answers"]:
            counter[normalize_answer(a["answer"])] += 1

    most_common = [ans for ans, _ in counter.most_common(top_k)]
    answer_to_idx = {ans: idx for idx, ans in enumerate(most_common)}
    idx_to_answer = {idx: ans for ans, idx in answer_to_idx.items()}
    return answer_to_idx, idx_to_answer


def get_image_transform(split="train"):
    """ImageNet-normalised transform for DINOv2 ViT-g/14 at 336x336 (24x24 patches + 1 CLS = 577 tokens).

    The HDF5 cache stores 336x336 images, so Resize(336) is a no-op on the short side
    and the downstream RandomCrop(336) / CenterCrop(336) selects the full image.
    DINOv2's 14-px patch grid * 24 = 336; position embeddings are interpolated by HF
    transparently when the input differs from the pretrained 14*16=224 grid.
    Horizontal flip is dropped: it inverts spatial language (left/right) used by
    many VQA questions.
    """
    if split == "train":
        return transforms.Compose([
            transforms.Resize(336),
            transforms.RandomCrop(336),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    return transforms.Compose([
        transforms.Resize(336),
        transforms.CenterCrop(336),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

def get_h5_preprocess_transform():
    """Resize to 336x336 for HDF5 storage (no normalization, no crop randomness)."""
    return transforms.Compose([
        transforms.Resize(336),
        transforms.CenterCrop(336),
    ])


## Storage as one .h5 file

resized to 256x256 before adding to .h5 file

In [10]:
# --- HDF5 Preprocessing: convert raw JPEGs into a single contiguous file ---
h5_path = DATA_DIR / "vqa_images_336.h5"

if not h5_path.exists():
    preprocess = get_h5_preprocess_transform()
    image_dirs = [DATA_DIR / "images" / "train2014", DATA_DIR / "images" / "val2014"]

    # Collect all image paths
    all_paths = []
    for d in image_dirs:
        if d.exists():
            all_paths.extend(sorted(d.glob("*.jpg")))
    print(f"Found {len(all_paths):,} images to preprocess")

    with h5py.File(h5_path, "w") as h5f:
        imgs_ds = h5f.create_dataset(
            "images", shape=(len(all_paths), 336, 336, 3),
            dtype=np.uint8, chunks=(1, 336, 336, 3))
        ids_ds = h5f.create_dataset(
            "image_ids", shape=(len(all_paths),), dtype=np.int64)

        for i, path in enumerate(tqdm(all_paths, desc="Preprocessing images")):
            image_id = int(path.stem.split("_")[-1])
            img = Image.open(path).convert("RGB")
            img = preprocess(img)
            imgs_ds[i] = np.array(img)
            ids_ds[i] = image_id

    print(f"Saved {len(all_paths):,} images to {h5_path}")
else:
    print(f"HDF5 file already exists: {h5_path}")


HDF5 file already exists: /content/data/vqa_images_336.h5


In [11]:
class VQADataset(Dataset):
    """PyTorch dataset for VQA v2.0 with HDF5 image loading and soft targets.

    Returns (image_tensor, input_ids, attention_mask, answer_target) tuples.
    answer_target is a soft probability vector of shape (num_answers,) built
    from all 10 annotator answers (after VQAv2-style normalization).
    """

    def __init__(self, questions_file, annotations_file, h5_path,
                 answer_to_idx=None, top_k_answers=3129,
                 max_question_len=20, transform=None, max_samples=None):
        self.h5_path = Path(h5_path)
        self.max_question_len = max_question_len
        self.transform = transform or get_image_transform("val")
        self.tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")
        self._h5f = None  # lazy-opened per DataLoader worker

        if answer_to_idx is None:
            self.answer_to_idx, self.idx_to_answer = build_answer_vocab(
                annotations_file, top_k_answers)
        else:
            self.answer_to_idx = answer_to_idx
            self.idx_to_answer = {v: k for k, v in answer_to_idx.items()}

        # Build image_id -> HDF5 row index mapping
        with h5py.File(self.h5_path, "r") as f:
            image_ids = f["image_ids"][:]
        self.id_to_row = {int(iid): i for i, iid in enumerate(image_ids)}

        with open(questions_file) as f:
            questions_data = json.load(f)["questions"]
        with open(annotations_file) as f:
            annotations_data = json.load(f)["annotations"]

        ann_by_qid = {ann["question_id"]: ann for ann in annotations_data}
        num_answers = len(self.answer_to_idx)

        self.samples = []
        for q in questions_data:
            ann = ann_by_qid.get(q["question_id"])
            if ann is None:
                continue

            # Soft target: target[ans_idx] = (# annotators who gave that answer) / 10.
            # Each annotator answer is normalized (lowercase, strip punctuation, drop
            # articles, fix contractions) before lookup so it matches the vocab keys.
            target = torch.zeros(num_answers, dtype=torch.float)
            for a in ann["answers"]:
                ans_text = normalize_answer(a["answer"])
                idx = self.answer_to_idx.get(ans_text)
                if idx is not None:
                    target[idx] += 1.0

            # Skip questions where no annotator answer falls within the top-k vocabulary
            if target.sum() == 0:
                continue

            target /= 10.0  # normalize: count -> fraction of annotators

            self.samples.append({
                "question": q["question"],
                "image_id": q["image_id"],
                "answer_target": target,
            })
            if max_samples is not None and len(self.samples) >= max_samples:
                break

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # Lazy HDF5 open (per-worker for multiprocessing safety)
        if self._h5f is None:
            self._h5f = h5py.File(self.h5_path, "r")

        row = self.id_to_row[sample["image_id"]]
        img_array = self._h5f["images"][row]  # (336, 336, 3) uint8
        image = Image.fromarray(img_array)
        image = self.transform(image)

        encoding = self.tokenizer(
            sample["question"],
            max_length=self.max_question_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        return image, input_ids, attention_mask, sample["answer_target"]


In [12]:
# Build answer vocab from training annotations
train_ann = DATA_DIR / "answers" / "v2_mscoco_train2014_annotations.json"
answer_to_idx, idx_to_answer = build_answer_vocab(train_ann, NUM_ANSWERS)
print(f"Answer vocab: {len(answer_to_idx)} classes")

# Create datasets (loading images from HDF5)
train_ds = VQADataset(
    questions_file=DATA_DIR / "questions" / "v2_OpenEnded_mscoco_train2014_questions.json",
    annotations_file=train_ann,
    h5_path=h5_path,
    answer_to_idx=answer_to_idx,
    max_question_len=MAX_QUESTION_LEN,
    transform=get_image_transform("train"),
    max_samples=MAX_SAMPLES,
)
val_ds = VQADataset(
    questions_file=DATA_DIR / "questions" / "v2_OpenEnded_mscoco_val2014_questions.json",
    annotations_file=DATA_DIR / "answers" / "v2_mscoco_val2014_annotations.json",
    h5_path=h5_path,
    answer_to_idx=answer_to_idx,
    max_question_len=MAX_QUESTION_LEN,
    transform=get_image_transform("val"),
    max_samples=MAX_SAMPLES,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_ds):,} samples ({len(train_loader)} batches)")
print(f"Val:   {len(val_ds):,} samples ({len(val_loader)} batches)")

# Save answer vocab
with open(CHECKPOINT_DIR / "answer_vocab.json", "w") as f:
    json.dump(answer_to_idx, f)

Answer vocab: 3129 classes


config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Train: 434,957 samples (27185 batches)
Val:   209,773 samples (13111 batches)


---
## 2. Model Definitions

### Encoders

Both encoders are **frozen** (pretrained weights fixed). Only the fusion layers and classifier train.

- **ImageEncoder** — DINOv2 ViT-g/14: produces 577 tokens (576 spatial + 1 CLS) at 336x336, projected from 1536 → `EMBED_DIM`
- **TextEncoder** — DeBERTa-v3-large: produces per-token embeddings, projected from 1024 → `EMBED_DIM`

In [13]:
class ImageEncoder(nn.Module):
    """DINOv2 ViT-g/14 image encoder. Output: (B, seq_len, embed_dim)."""

    VIT_HIDDEN_DIM = 1536

    def __init__(self, embed_dim=512, freeze=True):
        super().__init__()
        self.backbone = AutoModel.from_pretrained("facebook/dinov2-giant")
        # Required to fit the wider model (embed_dim=1024, fusion depth=6) in 24 GB VRAM.
        self.backbone.gradient_checkpointing_enable()
        # MLP projection: Linear -> LN -> GELU -> Linear -> Dropout. Gives the
        # encoder representations a learnable non-linearity into the shared
        # embedding space instead of a bare matrix multiply.
        self.projection = nn.Sequential(
            nn.Linear(self.VIT_HIDDEN_DIM, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, embed_dim),
            nn.Dropout(0.1),
        )

        if freeze:
            for param in self.backbone.parameters():
                param.requires_grad = False

    def forward(self, images):
        # 336x336 input -> 24*24 + 1 CLS = 577 tokens at hidden_dim=1536 (DINOv2 14-px patches; HF interpolates position embeddings).
        outputs = self.backbone(pixel_values=images)
        return self.projection(outputs.last_hidden_state)  # (B, seq_len, embed_dim)


class TextEncoder(nn.Module):
    """DeBERTa-v3-large text encoder. Output: (B, seq_len, embed_dim)."""

    # Name retained for backward compatibility with optimizer param-group filters
    # elsewhere in the notebook; value updated to DeBERTa-v3-large's hidden dim.
    ROBERTA_HIDDEN_DIM = 1024

    def __init__(self, embed_dim=512, freeze=True):
        super().__init__()
        # Attribute name `roberta` kept so any downstream `text_encoder.roberta.*`
        # access (e.g. parameter-group construction) continues to work unchanged.
        self.roberta = AutoModel.from_pretrained("microsoft/deberta-v3-large")
        self.roberta.gradient_checkpointing_enable()
        self.projection = nn.Sequential(
            nn.Linear(self.ROBERTA_HIDDEN_DIM, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, embed_dim),
            nn.Dropout(0.1),
        )

        if freeze:
            for param in self.roberta.parameters():
                param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        return self.projection(outputs.last_hidden_state)  # (B, seq_len, embed_dim)


### Cross-Attention Blocks

The core research contribution. Cross-attention lets one modality "ask questions" of the other:

- **Asymmetric**: Two **independent** blocks with separate weights — one for image→text, one for text→image
- **Symmetric** (baseline): A **single shared** block used in both directions — cannot learn directional patterns

In [14]:
class CrossAttentionBlock(nn.Module):
    """Cross-attention: queries from one modality attend to keys/values from another.
    Includes LayerNorm, residual connections, and a feed-forward network.

    Two dropout knobs:
      - attn_dropout: dropout INSIDE the multi-head attention softmax
      - dropout: dropout in the FFN sub-block
    """

    def __init__(self, embed_dim, num_heads=8, dropout=0.1, attn_dropout=None):
        super().__init__()
        if attn_dropout is None:
            attn_dropout = dropout
        # Pre-norm design: LayerNorm is applied BEFORE attention (not after).
        # This improves training stability for deep networks compared to post-norm.
        self.norm_q = nn.LayerNorm(embed_dim)
        self.norm_kv = nn.LayerNorm(embed_dim)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=attn_dropout, batch_first=True)
        self.norm_ff = nn.LayerNorm(embed_dim)
        # Standard Transformer FFN: expand 4x with GELU activation, then project back
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, query, key_value, key_padding_mask=None):
        # Pre-norm cross-attention: normalize inputs, attend, then ADD back the
        # original query (residual). This preserves the raw query signal while
        # enriching it with cross-modal context from key_value.
        q = self.norm_q(query)
        kv = self.norm_kv(key_value)
        need_weights = not self.training
        attended, attn_weights = self.cross_attn(
            q, kv, kv, key_padding_mask=key_padding_mask,
            need_weights=need_weights, average_attn_weights=True)  # weights saved for visualization
        query = query + attended  # residual connection
        # Pre-norm feed-forward + residual (same pattern)
        query = query + self.ff(self.norm_ff(query))
        return query, attn_weights


class SelfAttentionBlock(nn.Module):
    """Pre-norm transformer self-attention block (per-modality refinement)."""

    def __init__(self, embed_dim, num_heads, dropout, attn_dropout):
        super().__init__()
        if attn_dropout is None:
            attn_dropout = dropout
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads,
                                          dropout=attn_dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x, key_padding_mask=None):
        h = self.norm1(x)
        attn, _ = self.attn(h,h,h,key_padding_mask = key_padding_mask, need_weights = False)
        attn, _ = self.attn(h, h, h, key_padding_mask=key_padding_mask)
        x = x + attn
        x = x + self.ff(self.norm2(x))
        return x


class AttentionPooling(nn.Module):
    """Learned-query attention pool: a single query attends over the sequence.

    Replaces CLS-token pooling so the downstream classifier sees a learned
    weighted summary over all tokens (text padding ignored via key_padding_mask).
    """

    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x, key_padding_mask=None):
        q = self.query.expand(x.size(0), -1, -1)
        out, _ = self.attn(q, x, x, key_padding_mask=key_padding_mask, need_weights = False)
        return self.norm(out.squeeze(1))


class AsymmetricCrossModalFusion(nn.Module):
    """Asymmetric cross-modal fusion stacked `depth` times -- the core research contribution.

    Each layer:
      Step 1: per-stream self-attention refines within-modality structure
      Step 2: image attends to text             -> img  (visually-grounded image features)
      Step 3: text attends to the attended img  -> txt  (NOT the raw image!)
    Each direction has independently learned weights at every layer (= 2*depth cross blocks
    plus 2*depth self-attn blocks). The cross-layer information flow lets text representations
    leverage progressively-refined image-text grounding from prior layers.
    """

    def __init__(self, embed_dim, num_heads=8, dropout=0.1, attn_dropout=None, depth=FUSION_DEPTH):
        super().__init__()
        self.depth = depth
        self.sa_img = nn.ModuleList([
            SelfAttentionBlock(embed_dim, num_heads, dropout, attn_dropout)
            for _ in range(depth)
        ])
        self.sa_txt = nn.ModuleList([
            SelfAttentionBlock(embed_dim, num_heads, dropout, attn_dropout)
            for _ in range(depth)
        ])
        self.i2t_layers = nn.ModuleList([
            CrossAttentionBlock(embed_dim, num_heads, dropout, attn_dropout)
            for _ in range(depth)
        ])
        self.t2i_layers = nn.ModuleList([
            CrossAttentionBlock(embed_dim, num_heads, dropout, attn_dropout)
            for _ in range(depth)
        ])

    def forward(self, image_features, text_features, text_padding_mask=None):
        img, txt = image_features, text_features
        last_i2t = last_t2i = None
        for i in range(self.depth):
            img = self.sa_img[i](img)
            txt = self.sa_txt[i](txt, key_padding_mask=text_padding_mask)
            img, last_i2t = self.i2t_layers[i](
                query=img, key_value=txt, key_padding_mask=text_padding_mask)
            txt, last_t2i = self.t2i_layers[i](query=txt, key_value=img)
        # Return last-layer attention so visualization cells stay unchanged.
        return img, txt, last_i2t, last_t2i


class SymmetricCrossModalFusion(nn.Module):
    """Symmetric cross-modal fusion stacked `depth` times (baseline).

    Per-stream self-attention has independent weights per layer (same as asymmetric).
    The only weight-sharing difference vs. asymmetric is in the cross-attention block:
    a single shared CrossAttentionBlock per layer is used for both i2t and t2i
    directions in parallel on the previous layer's outputs.
    """

    def __init__(self, embed_dim, num_heads=8, dropout=0.1, attn_dropout=None, depth=FUSION_DEPTH):
        super().__init__()
        self.depth = depth
        self.sa_img = nn.ModuleList([
            SelfAttentionBlock(embed_dim, num_heads, dropout, attn_dropout)
            for _ in range(depth)
        ])
        self.sa_txt = nn.ModuleList([
            SelfAttentionBlock(embed_dim, num_heads, dropout, attn_dropout)
            for _ in range(depth)
        ])
        self.shared_layers = nn.ModuleList([
            CrossAttentionBlock(embed_dim, num_heads, dropout, attn_dropout)
            for _ in range(depth)
        ])

    def forward(self, image_features, text_features, text_padding_mask=None):
        img, txt = image_features, text_features
        last_i2t = last_t2i = None
        for i in range(self.depth):
            img = self.sa_img[i](img)
            txt = self.sa_txt[i](txt, key_padding_mask=text_padding_mask)
            shared = self.shared_layers[i]
            new_img, last_i2t = shared(
                query=img, key_value=txt, key_padding_mask=text_padding_mask)
            new_txt, last_t2i = shared(query=txt, key_value=img)
            img, txt = new_img, new_txt
        return img, txt, last_i2t, last_t2i


### Full VQA Models

Pipeline: encode image + text → cross-modal fusion → mean-pool → concatenate → MLP classifier

The two models are identical except for the fusion layer.

In [15]:
class AsymmetricVQAModel(nn.Module):
    """VQA model with asymmetric (two independent blocks) cross-attention.
    Accepts pre-extracted encoder features -- no internal encoders."""

    def __init__(self, num_answers, embed_dim=512, num_heads=8, dropout=0.3,
                 attn_dropout=None, cls_dropout=None):
        super().__init__()
        if cls_dropout is None:
            cls_dropout = dropout
        self.fusion = AsymmetricCrossModalFusion(embed_dim, num_heads, dropout, attn_dropout)
        self.pool_img = AttentionPooling(embed_dim, num_heads)
        self.pool_txt = AttentionPooling(embed_dim, num_heads)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim * 2),
            nn.LayerNorm(embed_dim * 2),
            nn.GELU(),
            nn.Dropout(cls_dropout),
            nn.Linear(embed_dim * 2, num_answers),
        )

    def forward(self, image_features, text_features, attention_mask):
        text_pad_mask = attention_mask == 0
        img_att, txt_att, attn_i2t, attn_t2i = self.fusion(
            image_features, text_features, text_pad_mask)
        pooled_img = self.pool_img(img_att)
        pooled_txt = self.pool_txt(txt_att, key_padding_mask=text_pad_mask)
        z = torch.cat([pooled_img, pooled_txt], dim=-1)
        logits = self.classifier(z)
        return logits, {"img_to_txt": attn_i2t, "txt_to_img": attn_t2i}


class SymmetricVQAModel(nn.Module):
    """VQA model with symmetric (single shared block) cross-attention (baseline).
    Accepts pre-extracted encoder features -- no internal encoders."""

    def __init__(self, num_answers, embed_dim=512, num_heads=8, dropout=0.3,
                 attn_dropout=None, cls_dropout=None):
        super().__init__()
        if cls_dropout is None:
            cls_dropout = dropout
        self.fusion = SymmetricCrossModalFusion(embed_dim, num_heads, dropout, attn_dropout)
        self.pool_img = AttentionPooling(embed_dim, num_heads)
        self.pool_txt = AttentionPooling(embed_dim, num_heads)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim * 2),
            nn.LayerNorm(embed_dim * 2),
            nn.GELU(),
            nn.Dropout(cls_dropout),
            nn.Linear(embed_dim * 2, num_answers),
        )

    def forward(self, image_features, text_features, attention_mask):
        text_pad_mask = attention_mask == 0
        img_att, txt_att, attn_i2t, attn_t2i = self.fusion(
            image_features, text_features, text_pad_mask)
        pooled_img = self.pool_img(img_att)
        pooled_txt = self.pool_txt(txt_att, key_padding_mask=text_pad_mask)
        z = torch.cat([pooled_img, pooled_txt], dim=-1)
        logits = self.classifier(z)
        return logits, {"img_to_txt": attn_i2t, "txt_to_img": attn_t2i}


# --- End-to-end models (used when FREEZE_ENCODERS=False) ---

class AsymmetricVQAModelE2E(nn.Module):
    """End-to-end VQA model: encoders + asymmetric fusion + classifier."""

    def __init__(self, num_answers, embed_dim=512, num_heads=8, dropout=0.3,
                 freeze_encoders=False, attn_dropout=None, cls_dropout=None):
        super().__init__()
        if cls_dropout is None:
            cls_dropout = dropout
        self.image_encoder = ImageEncoder(embed_dim, freeze=freeze_encoders)
        self.text_encoder = TextEncoder(embed_dim, freeze=freeze_encoders)
        self.fusion = AsymmetricCrossModalFusion(embed_dim, num_heads, dropout, attn_dropout)
        self.pool_img = AttentionPooling(embed_dim, num_heads)
        self.pool_txt = AttentionPooling(embed_dim, num_heads)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim * 2),
            nn.LayerNorm(embed_dim * 2),
            nn.GELU(),
            nn.Dropout(cls_dropout),
            nn.Linear(embed_dim * 2, num_answers),
        )

    def forward(self, images, input_ids, attention_mask):
        img = self.image_encoder(images)
        txt = self.text_encoder(input_ids, attention_mask)
        text_pad_mask = attention_mask == 0
        img_att, txt_att, attn_i2t, attn_t2i = self.fusion(img, txt, text_pad_mask)
        pooled_img = self.pool_img(img_att)
        pooled_txt = self.pool_txt(txt_att, key_padding_mask=text_pad_mask)
        z = torch.cat([pooled_img, pooled_txt], dim=-1)
        logits = self.classifier(z)
        return logits, {"img_to_txt": attn_i2t, "txt_to_img": attn_t2i}


class SymmetricVQAModelE2E(nn.Module):
    """End-to-end VQA model: encoders + symmetric fusion + classifier."""

    def __init__(self, num_answers, embed_dim=512, num_heads=8, dropout=0.3,
                 freeze_encoders=False, attn_dropout=None, cls_dropout=None):
        super().__init__()
        if cls_dropout is None:
            cls_dropout = dropout
        self.image_encoder = ImageEncoder(embed_dim, freeze=freeze_encoders)
        self.text_encoder = TextEncoder(embed_dim, freeze=freeze_encoders)
        self.fusion = SymmetricCrossModalFusion(embed_dim, num_heads, dropout, attn_dropout)
        self.pool_img = AttentionPooling(embed_dim, num_heads)
        self.pool_txt = AttentionPooling(embed_dim, num_heads)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim * 2),
            nn.LayerNorm(embed_dim * 2),
            nn.GELU(),
            nn.Dropout(cls_dropout),
            nn.Linear(embed_dim * 2, num_answers),
        )

    def forward(self, images, input_ids, attention_mask):
        img = self.image_encoder(images)
        txt = self.text_encoder(input_ids, attention_mask)
        text_pad_mask = attention_mask == 0
        img_att, txt_att, attn_i2t, attn_t2i = self.fusion(img, txt, text_pad_mask)
        pooled_img = self.pool_img(img_att)
        pooled_txt = self.pool_txt(txt_att, key_padding_mask=text_pad_mask)
        z = torch.cat([pooled_img, pooled_txt], dim=-1)
        logits = self.classifier(z)
        return logits, {"img_to_txt": attn_i2t, "txt_to_img": attn_t2i}


### Offline Feature ExtractionSince encoders are frozen (`FREEZE_ENCODERS = True`), every sample produces identical features every epoch.
We extract once and cache to HDF5, eliminating redundant ViT + RoBERTa forward passes.
**Tradeoffs:**- Train augmentation (RandomCrop, RandomHorizontalFlip) is replaced with deterministic CenterCrop- Projection layers (768→512) are baked into features (reproducible via fixed seed)- Storage: ~140 GB float16 for the full dataset

In [16]:
FEATURES_H5 = DATA_DIR / "vqa_precomputed_features.h5"

if FREEZE_ENCODERS:
    @torch.no_grad()
    def extract_and_save_features(image_encoder, text_encoder, dataset, split_name, h5_path,
                                   batch_size=BATCH_SIZE, num_workers=NUM_WORKERS):
        """Run frozen encoders once over the dataset and save features to HDF5."""
        image_encoder.eval()
        text_encoder.eval()

        # Use deterministic val transform for reproducible extraction
        orig_transform = dataset.transform
        dataset.transform = get_image_transform("val")

        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                            num_workers=num_workers, pin_memory=True)

        n = len(dataset)
        use_amp = USE_AMP and device.type == "cuda"

        # Estimate disk usage
        bytes_per_sample = (577 * EMBED_DIM + MAX_QUESTION_LEN * EMBED_DIM) * 2  # float16
        bytes_per_sample += MAX_QUESTION_LEN * 1 + NUM_ANSWERS * 4               # int8 + float32
        est_gb = n * bytes_per_sample / 1e9
        print(f"  Extracting {split_name}: {n:,} samples, estimated {est_gb:.1f} GB")

        with h5py.File(h5_path, "a") as h5f:
            img_ds  = h5f.create_dataset(f"{split_name}/image_features",
                                         shape=(n, 577, EMBED_DIM), dtype="float16")
            txt_ds  = h5f.create_dataset(f"{split_name}/text_features",
                                         shape=(n, MAX_QUESTION_LEN, EMBED_DIM), dtype="float16")
            mask_ds = h5f.create_dataset(f"{split_name}/attention_mask",
                                         shape=(n, MAX_QUESTION_LEN), dtype="int8")
            ans_ds  = h5f.create_dataset(f"{split_name}/answer_target",
                                         shape=(n, NUM_ANSWERS), dtype="float32")

            idx = 0
            for images, input_ids, attention_mask, answers in tqdm(loader, desc=f"  {split_name}"):
                images = images.to(device)
                input_ids = input_ids.to(device)
                attn_mask_dev = attention_mask.to(device)

                amp_ctx = torch.amp.autocast(device_type=device.type, dtype=AMP_DTYPE) if use_amp else nullcontext()
                with amp_ctx:
                    img_feats = image_encoder(images)                       # (B, 577, EMBED_DIM)
                    txt_feats = text_encoder(input_ids, attn_mask_dev)      # (B, 20, EMBED_DIM)

                bs = images.size(0)
                img_ds[idx:idx+bs]  = img_feats.cpu().half().numpy()
                txt_ds[idx:idx+bs]  = txt_feats.cpu().half().numpy()
                mask_ds[idx:idx+bs] = attention_mask.numpy().astype("int8")
                ans_ds[idx:idx+bs]  = answers.numpy()
                idx += bs

        dataset.transform = orig_transform
        print(f"  {split_name}: {n:,} samples saved.")

    # --- Run extraction ---
    set_seed(SEED)
    _img_enc = ImageEncoder(EMBED_DIM, freeze=True).to(device)
    _txt_enc = TextEncoder(EMBED_DIM, freeze=True).to(device)

    if FEATURES_H5.exists():
        FEATURES_H5.unlink()

    extract_and_save_features(_img_enc, _txt_enc, train_ds, "train", FEATURES_H5)
    extract_and_save_features(_img_enc, _txt_enc, val_ds,   "val",   FEATURES_H5)

    del _img_enc, _txt_enc
    torch.cuda.empty_cache()

    # Verify
    with h5py.File(FEATURES_H5, "r") as f:
        for split in ["train", "val"]:
            print(f"  {split}: image_features={f[f'{split}/image_features'].shape}, "
                  f"text_features={f[f'{split}/text_features'].shape}")
else:
    print("FREEZE_ENCODERS=False: skipping offline feature extraction.")
    print("Training will use raw images/tokens with end-to-end models.")

FREEZE_ENCODERS=False: skipping offline feature extraction.
Training will use raw images/tokens with end-to-end models.


### Precomputed Feature DatasetReads pre-extracted encoder features directly from HDF5.Bypasses JPEG loading, PIL transforms, and RoBERTa tokenization entirely.

In [17]:
if FREEZE_ENCODERS:
    class PrecomputedVQADataset(Dataset):
        """Dataset that reads pre-extracted encoder features from HDF5."""

        def __init__(self, h5_path, split):
            self.h5_path = str(h5_path)
            self.split = split
            self._h5f = None  # lazy-opened per DataLoader worker

            with h5py.File(self.h5_path, "r") as f:
                self.n = f[f"{split}/image_features"].shape[0]

        def __len__(self):
            return self.n

        def __getitem__(self, idx):
            if self._h5f is None:
                self._h5f = h5py.File(self.h5_path, "r")

            img_feats = torch.from_numpy(
                self._h5f[f"{self.split}/image_features"][idx].astype("float32"))   # (577, EMBED_DIM)
            txt_feats = torch.from_numpy(
                self._h5f[f"{self.split}/text_features"][idx].astype("float32"))    # (20, EMBED_DIM)
            att_mask = torch.from_numpy(
                self._h5f[f"{self.split}/attention_mask"][idx].astype("int64"))     # (20,)
            answer = torch.from_numpy(
                self._h5f[f"{self.split}/answer_target"][idx])                      # (NUM_ANSWERS,)

            return img_feats, txt_feats, att_mask, answer

    # Shadow the original datasets and loaders with precomputed versions
    train_ds = PrecomputedVQADataset(FEATURES_H5, "train")
    val_ds   = PrecomputedVQADataset(FEATURES_H5, "val")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

    print(f"Train: {len(train_ds):,} samples ({len(train_loader)} batches)")
    print(f"Val:   {len(val_ds):,} samples ({len(val_loader)} batches)")

    # Sanity check: verify shapes from one batch
    _img, _txt, _mask, _ans = next(iter(val_loader))
    print(f"Batch shapes: img={_img.shape}, txt={_txt.shape}, mask={_mask.shape}, ans={_ans.shape}")
    del _img, _txt, _mask, _ans
else:
    # Rebuild loaders with the (smaller) BATCH_SIZE for end-to-end training
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

    print(f"End-to-end mode (batch_size={BATCH_SIZE})")
    print(f"Train: {len(train_ds):,} samples ({len(train_loader)} batches)")
    print(f"Val:   {len(val_ds):,} samples ({len(val_loader)} batches)")

End-to-end mode (batch_size=16)
Train: 434,957 samples (27185 batches)
Val:   209,773 samples (13111 batches)


---
## 3. Training

In [18]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, use_amp, scheduler=None):
    model.train()
    optimizer.zero_grad()
    total_loss = 0.0
    correct = 0
    total = 0
    n_batches = len(loader)

    for batch_idx, (inp1, inp2, masks, answers) in enumerate(tqdm(loader, desc="  train", leave=False)):
        inp1    = inp1.to(device)
        inp2    = inp2.to(device)
        masks   = masks.to(device)
        answers = answers.to(device)

        amp_ctx = torch.amp.autocast(device_type=device.type, dtype=AMP_DTYPE) if use_amp else nullcontext()
        with amp_ctx:
            logits, _ = model(inp1, inp2, masks)
            loss = criterion(logits, answers)
            loss_to_back = loss / GRAD_ACCUM_STEPS

        if scaler is not None:
            scaler.scale(loss_to_back).backward()
        else:
            loss_to_back.backward()

        is_step_boundary = ((batch_idx + 1) % GRAD_ACCUM_STEPS == 0) or (batch_idx + 1 == n_batches)
        if is_step_boundary:
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad()
            if scheduler is not None:
                scheduler.step()

        total_loss += loss.item() * answers.size(0)
        correct += (logits.argmax(dim=1) == answers.argmax(dim=1)).sum().item()
        total += answers.size(0)

    return {"train_loss": total_loss / total, "train_acc": correct / total * 100}


@torch.no_grad()
def evaluate(model, loader, criterion, use_amp):
    """Evaluate using the official VQA accuracy metric: min(1, count/3)."""
    model.eval()
    total_loss = 0.0
    vqa_acc_sum = 0.0
    total = 0

    for inp1, inp2, masks, answers in tqdm(loader, desc="  eval", leave=False):
        inp1    = inp1.to(device)
        inp2    = inp2.to(device)
        masks   = masks.to(device)
        answers = answers.to(device)

        amp_ctx = torch.amp.autocast(device_type=device.type, dtype=AMP_DTYPE) if use_amp else nullcontext()
        with amp_ctx:
            logits, _ = model(inp1, inp2, masks)
            loss = criterion(logits, answers)

        total_loss += loss.item() * answers.size(0)

        preds = logits.argmax(dim=1)
        pred_soft = answers[torch.arange(answers.size(0), device=answers.device), preds]
        vqa_scores = torch.clamp(pred_soft * 10.0 / 3.0, max=1.0)
        vqa_acc_sum += vqa_scores.sum().item()
        total += answers.size(0)

    return {
        "val_loss": total_loss / total,
        "val_vqa_acc": vqa_acc_sum / total * 100,
    }


In [19]:
def run_training(model_type, run_name=None, resume_from=None):
    """Train a VQA model or resume from a checkpoint and return (model, history)."""
    set_seed(SEED)
    if run_name is None:
        run_name = f"{model_type}_s{SEED}"

    # --- Build model ---
    common_kwargs = dict(attn_dropout=ATTN_DROPOUT, cls_dropout=CLS_DROPOUT)
    if FREEZE_ENCODERS:
        if model_type == "asymmetric":
            model = AsymmetricVQAModel(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT, **common_kwargs)
        else:
            model = SymmetricVQAModel(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT, **common_kwargs)
    else:
        if model_type == "asymmetric":
            model = AsymmetricVQAModelE2E(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT,
                                          freeze_encoders=False, **common_kwargs)
        else:
            model = SymmetricVQAModelE2E(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT,
                                          freeze_encoders=False, **common_kwargs)
    model = model.to(device)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\nModel: {model_type} | Trainable params: {trainable:,} / {total_params:,} total")

    # --- Optimizer + Scheduler ---
    criterion = nn.BCEWithLogitsLoss()

    # Exclude bias and LayerNorm parameters from weight decay -- standard transformer recipe.
    NO_DECAY_KEYS = ("bias", "LayerNorm.weight", "layer_norm.weight", "ln_")
    def _split_named(named):
        decay, nodecay = [], []
        for n, p in named:
            (nodecay if any(k in n for k in NO_DECAY_KEYS) else decay).append(p)
        return decay, nodecay

    if FREEZE_ENCODERS:
        named = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
        decay, nodecay = _split_named(named)
        groups = []
        if decay:   groups.append({"params": decay,   "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY})
        if nodecay: groups.append({"params": nodecay, "lr": LEARNING_RATE, "weight_decay": 0.0})
        optimizer = bnb.optim.AdamW8but(groups)
        scheduler = None
    else:
        # --- LLRD parameter groups: per-layer LR for encoders, flat LR for fusion+cls ---
        groups = []

        # Text encoder: DeBERTa-v3-large, 24 layers
        TEXT_LAYERS = 24
        for i in range(TEXT_LAYERS):
            prefix = f"text_encoder.roberta.encoder.layer.{i}."
            layer_named = [(n, p) for n, p in model.named_parameters()
                           if p.requires_grad and n.startswith(prefix)]
            if not layer_named:
                continue
            d, nd = _split_named(layer_named)
            lr_i = ENCODER_LR * (LLRD_DECAY ** (TEXT_LAYERS - i))
            if d:  groups.append({"params": d,  "lr": lr_i, "weight_decay": WEIGHT_DECAY})
            if nd: groups.append({"params": nd, "lr": lr_i, "weight_decay": 0.0})

        text_emb = [(n, p) for n, p in model.named_parameters()
                    if p.requires_grad and n.startswith("text_encoder.roberta.embeddings.")]
        if text_emb:
            d, nd = _split_named(text_emb)
            lr_emb = ENCODER_LR * (LLRD_DECAY ** (TEXT_LAYERS + 1))
            if d:  groups.append({"params": d,  "lr": lr_emb, "weight_decay": WEIGHT_DECAY})
            if nd: groups.append({"params": nd, "lr": lr_emb, "weight_decay": 0.0})

        # Image encoder: DINOv2-giant, 40 blocks
        IMG_LAYERS = 40
        for i in range(IMG_LAYERS):
            prefix = f"image_encoder.backbone.encoder.layer.{i}."
            layer_named = [(n, p) for n, p in model.named_parameters()
                           if p.requires_grad and n.startswith(prefix)]
            if not layer_named:
                continue
            d, nd = _split_named(layer_named)
            lr_i = ENCODER_LR * (LLRD_DECAY ** (IMG_LAYERS - i))
            if d:  groups.append({"params": d,  "lr": lr_i, "weight_decay": WEIGHT_DECAY})
            if nd: groups.append({"params": nd, "lr": lr_i, "weight_decay": 0.0})

        img_emb = [(n, p) for n, p in model.named_parameters()
                   if p.requires_grad and n.startswith("image_encoder.backbone.embeddings.")]
        if img_emb:
            d, nd = _split_named(img_emb)
            lr_emb = ENCODER_LR * (LLRD_DECAY ** (IMG_LAYERS + 1))
            if d:  groups.append({"params": d,  "lr": lr_emb, "weight_decay": WEIGHT_DECAY})
            if nd: groups.append({"params": nd, "lr": lr_emb, "weight_decay": 0.0})

        # Leftover encoder params (e.g. top-level LayerNorm on the backbone) at ENCODER_LR
        def _in_llrd(name):
            return (name.startswith("text_encoder.roberta.encoder.layer.")
                    or name.startswith("text_encoder.roberta.embeddings.")
                    or name.startswith("image_encoder.backbone.encoder.layer.")
                    or name.startswith("image_encoder.backbone.embeddings."))

        leftover_enc = [(n, p) for n, p in model.named_parameters()
                        if p.requires_grad
                        and (n.startswith("text_encoder.roberta.")
                             or n.startswith("image_encoder.backbone."))
                        and not _in_llrd(n)]
        if leftover_enc:
            d, nd = _split_named(leftover_enc)
            if d:  groups.append({"params": d,  "lr": ENCODER_LR, "weight_decay": WEIGHT_DECAY})
            if nd: groups.append({"params": nd, "lr": ENCODER_LR, "weight_decay": 0.0})

        # Fusion + classifier + encoder projection heads at LEARNING_RATE.
        # Projection heads (image_encoder.projection.*, text_encoder.projection.*) fall
        # through here because they don't start with image_encoder.backbone. / text_encoder.roberta.
        other_named = [(n, p) for n, p in model.named_parameters()
                       if p.requires_grad
                       and not n.startswith("text_encoder.roberta.")
                       and not n.startswith("image_encoder.backbone.")]
        od, ond = _split_named(other_named)
        if od:  groups.append({"params": od,  "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY})
        if ond: groups.append({"params": ond, "lr": LEARNING_RATE, "weight_decay": 0.0})

        # Sanity check: every trainable param is in exactly one group.
        accounted = sum(p.numel() for g in groups for p in g["params"])
        assert accounted == trainable, (
            f"LLRD param-group construction missed/duplicated params: "
            f"{accounted:,} grouped vs {trainable:,} trainable")
        print(f"  LLRD groups: {len(groups)} | top-layer encoder lr={ENCODER_LR}, "
              f"bottom-layer encoder lr={ENCODER_LR * (LLRD_DECAY ** 41):.2e}, "
              f"fusion+cls lr={LEARNING_RATE}")

        optimizer = bnb.optim.AdamW8bit(groups)

        # Linear warmup, then cosine decay to ~0 over the remaining epochs.
        # Step counts account for gradient accumulation (one optimizer step per
        # GRAD_ACCUM_STEPS micro-batches).
        import math as _math
        steps_per_epoch = max(1, len(train_loader) // GRAD_ACCUM_STEPS)
        total_steps  = steps_per_epoch * EPOCHS
        warmup_steps = steps_per_epoch * WARMUP_EPOCHS

        def lr_lambda(step):
            if step < warmup_steps:
                return step / max(1, warmup_steps)
            progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
            return 0.5 * (1.0 + _math.cos(_math.pi * progress))

        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    use_amp = USE_AMP and device.type == "cuda"
    # GradScaler only needed for fp16 (which has limited dynamic range). bf16 / fp32 don't need it.
    scaler = GradScaler() if (use_amp and AMP_DTYPE == torch.float16) else None

    history = []
    best_val_acc = 0.0
    best_acc_epoch = None  # epoch whose _epoch{N}.pt currently holds best val_vqa_acc
    start_epoch = 1

    # --- RESUME LOGIC ---
    if resume_from is not None:
        checkpoint_path = Path(resume_from)
        if checkpoint_path.exists():
            print(f"Resuming from checkpoint: {checkpoint_path}")
            checkpoint = torch.load(checkpoint_path, map_location=device)
            try:
                model.load_state_dict(checkpoint["model_state_dict"], strict=True)
                optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
                if scheduler is not None and checkpoint.get("scheduler_state_dict") is not None:
                    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
                start_epoch = checkpoint["epoch"] + 1

                # Load previous history to keep appending
                history_file = METRICS_DIR / f"{run_name}_history.json"
                if history_file.exists():
                    with open(history_file, "r") as f:
                        history = json.load(f)

                if history:
                    best_entry = max(history, key=lambda h: h.get("val_vqa_acc", 0))
                    best_val_acc = best_entry.get("val_vqa_acc", 0)
                    best_acc_epoch = best_entry["epoch"]
            except (RuntimeError, ValueError) as e:
                # Architecture change: shapes or keys don't match. Start fresh.
                first_line = str(e).splitlines()[0] if str(e) else ""
                print(f"  Checkpoint at {checkpoint_path.name} is incompatible with the "
                      f"current architecture and will be ignored. Starting fresh.\n"
                      f"  ({e.__class__.__name__}: {first_line})")
                start_epoch = 1
                history = []
                best_val_acc = 0.0
                best_acc_epoch = None
        else:
            print(f"Checkpoint not found at {checkpoint_path}. Starting from scratch.")

    # --- TRAINING LOOP ---
    for epoch in range(start_epoch, EPOCHS + 1):
        t0 = time.time()
        train_m = train_one_epoch(model, train_loader, criterion, optimizer, scaler, use_amp,
                                  scheduler=scheduler)
        val_m = evaluate(model, val_loader, criterion, use_amp)
        elapsed = time.time() - t0

        epoch_data = {"epoch": epoch, **train_m, **val_m, "elapsed_s": round(elapsed, 1)}
        history.append(epoch_data)

        print(f"  Epoch {epoch}/{EPOCHS} | loss {train_m['train_loss']:.4f} | "
              f"train {train_m['train_acc']:.2f}% | vqa_acc {val_m['val_vqa_acc']:.2f}% | {elapsed:.0f}s")

        epoch_ckpt = CHECKPOINT_DIR / f"{run_name}_epoch{epoch}.pt"
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "metrics": epoch_data,
        }, epoch_ckpt)

        if val_m["val_vqa_acc"] > best_val_acc:
            best_val_acc = val_m["val_vqa_acc"]
            best_acc_epoch = epoch
            torch.save({"model_state_dict": model.state_dict(), **epoch_data},
                       CHECKPOINT_DIR / f"{run_name}_best.pt")
            print(f"    New best: {best_val_acc:.2f}%")

        # Cap disk usage: keep only the latest epoch and the best-so-far epoch.
        # _best.pt has weights only -- keeping the best epoch file preserves its
        # optimizer state so we can resume from a known-good point.
        keep = {epoch_ckpt}
        if best_acc_epoch is not None:
            keep.add(CHECKPOINT_DIR / f"{run_name}_epoch{best_acc_epoch}.pt")
        for stale in CHECKPOINT_DIR.glob(f"{run_name}_epoch*.pt"):
            if stale not in keep:
                stale.unlink(missing_ok=True)

        # Persist history after every epoch so a mid-run crash still leaves a
        # readable file for the visualization cells and the resume path.
        with open(METRICS_DIR / f"{run_name}_history.json", "w") as f:
            json.dump(history, f, indent=2)

    print(f"Training complete. Best val_vqa_acc: {best_val_acc:.2f}%")
    return model, history


### 3.1 Train Symmetric Baseline

In [20]:
# Auto-resume: if any symmetric_s42_epoch*.pt exists, pick the highest epoch
# and pass it as resume_from so re-running this cell continues training.
# Before resuming, verify the checkpoint matches the current architecture --
# stale checkpoints from earlier configs are silently skipped.
# sym_ckpts = sorted(
#     CHECKPOINT_DIR.glob("symmetric_s42_epoch*.pt"),
#     key=lambda p: int(p.stem.rsplit("epoch", 1)[1]),
# )
# sym_resume = sym_ckpts[-1] if sym_ckpts else None
# if sym_resume is not None:
#     _probe = (SymmetricVQAModelE2E(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT,
#                                    freeze_encoders=False,
#                                    attn_dropout=ATTN_DROPOUT, cls_dropout=CLS_DROPOUT)
#               if not FREEZE_ENCODERS else
#               SymmetricVQAModel(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT,
#                                 attn_dropout=ATTN_DROPOUT, cls_dropout=CLS_DROPOUT))
#     _ckpt_sd  = torch.load(sym_resume, map_location="cpu")["model_state_dict"]
#     _probe_sd = _probe.state_dict()
#     _shape_bad = any(k in _probe_sd and _probe_sd[k].shape != _ckpt_sd[k].shape
#                      for k in _ckpt_sd)
#     _key_diff = set(_probe_sd) ^ set(_ckpt_sd)
#     if _shape_bad or _key_diff:
#         print(f"Skipping resume from {sym_resume.name}: architecture changed "
#               f"(shape_mismatch={_shape_bad}, key_diff={len(_key_diff)}). Training fresh.")
#         sym_resume = None
#     del _probe, _ckpt_sd, _probe_sd
# symmetric_model, symmetric_history = run_training("symmetric", resume_from=sym_resume)


### 3.2 Train Asymmetric Model

In [21]:
# Auto-resume: pick the highest-numbered asymmetric_s42_epoch*.pt if any.
# Before resuming, verify the checkpoint matches the current architecture --
# stale checkpoints from earlier configs are silently skipped.
asym_ckpts = sorted(
    CHECKPOINT_DIR.glob("asymmetric_s42_epoch*.pt"),
    key=lambda p: int(p.stem.rsplit("epoch", 1)[1]),
)
asym_resume = asym_ckpts[-1] if asym_ckpts else None
if asym_resume is not None:
    _probe = (AsymmetricVQAModelE2E(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT,
                                    freeze_encoders=False,
                                    attn_dropout=ATTN_DROPOUT, cls_dropout=CLS_DROPOUT)
              if not FREEZE_ENCODERS else
              AsymmetricVQAModel(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT,
                                 attn_dropout=ATTN_DROPOUT, cls_dropout=CLS_DROPOUT))
    _ckpt_sd  = torch.load(asym_resume, map_location="cpu")["model_state_dict"]
    _probe_sd = _probe.state_dict()
    _shape_bad = any(k in _probe_sd and _probe_sd[k].shape != _ckpt_sd[k].shape
                     for k in _ckpt_sd)
    _key_diff = set(_probe_sd) ^ set(_ckpt_sd)
    if _shape_bad or _key_diff:
        print(f"Skipping resume from {asym_resume.name}: architecture changed "
              f"(shape_mismatch={_shape_bad}, key_diff={len(_key_diff)}). Training fresh.")
        asym_resume = None
    del _probe, _ckpt_sd, _probe_sd
asymmetric_model, asymmetric_history = run_training("asymmetric", resume_from=asym_resume)


config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

pytorch_model.bin:   0%|          | 0.00/874M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/874M [00:00<?, ?B/s]


Model: asymmetric | Trainable params: 1,896,568,377 / 1,896,568,377 total
  LLRD groups: 136 | top-layer encoder lr=1e-05, bottom-layer encoder lr=1.22e-06, fusion+cls lr=0.0001



  train:   0%|          | 96/27185 [01:51<8:31:56,  1.13s/it]
                                                             

KeyboardInterrupt: 

### 3.3 Load Models from .pt File

In [ ]:
# # Rebuild the model architecture (match the training-time construction)
# _common = dict(attn_dropout=ATTN_DROPOUT, cls_dropout=CLS_DROPOUT)
# if FREEZE_ENCODERS:
#     symmetric_model = SymmetricVQAModel(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT, **_common).to(device)
# else:
#     symmetric_model = SymmetricVQAModelE2E(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT,
#                                             freeze_encoders=False, **_common).to(device)
#
# sym_checkpoint = torch.load(CHECKPOINT_DIR / "symmetric_s42_best.pt", map_location=device)
# symmetric_model.load_state_dict(sym_checkpoint["model_state_dict"])
# symmetric_model.eval()
#
# sym_history_file = METRICS_DIR / "symmetric_s42_history.json"
# if sym_history_file.exists():
#     with open(sym_history_file, "r") as f:
#         symmetric_history = json.load(f)
# else:
#     symmetric_history = []
#     print(f"WARNING: {sym_history_file.name} missing — training did not complete.")


In [ ]:
# Rebuild the model architecture (match the training-time construction in run_training)
_common = dict(attn_dropout=ATTN_DROPOUT, cls_dropout=CLS_DROPOUT)
if FREEZE_ENCODERS:
    asymmetric_model = AsymmetricVQAModel(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT, **_common).to(device)
else:
    asymmetric_model = AsymmetricVQAModelE2E(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT,
                                              freeze_encoders=False, **_common).to(device)

asym_checkpoint = torch.load(CHECKPOINT_DIR / "asymmetric_s42_best.pt", map_location=device)
asymmetric_model.load_state_dict(asym_checkpoint["model_state_dict"])
asymmetric_model.eval()

# Load its history so the graphs plot correctly
asym_history_file = METRICS_DIR / "asymmetric_s42_history.json"
if asym_history_file.exists():
    with open(asym_history_file, "r") as f:
        asymmetric_history = json.load(f)
else:
    asymmetric_history = []
    print(f"WARNING: {asym_history_file.name} missing — training did not complete.")


---
## 4. Evaluation

In [ ]:
criterion = nn.BCEWithLogitsLoss()
use_amp = USE_AMP and device.type == "cuda"

# sym_metrics = evaluate(symmetric_model, val_loader, criterion, use_amp)
asym_metrics = evaluate(asymmetric_model, val_loader, criterion, use_amp)

# results = {"Symmetric": sym_metrics, "Asymmetric": asym_metrics}
results = {"Asymmetric": asym_metrics}

# Print comparison table
metrics_keys = ["val_vqa_acc", "val_loss"]
header = "| Method       | " + " | ".join(k.replace('_', ' ').title() for k in metrics_keys) + " |"
sep    = "|--------------|" + "|".join("----------" for _ in metrics_keys) + "|"
print(header)
print(sep)
for name, vals in results.items():
    row = f"| {name:<12} | " + " | ".join(f"{vals[k]:>8.2f}" for k in metrics_keys) + " |"
    print(row)

---
## 5. Visualization

### 5.1 Training Curves

In [ ]:
# histories = {"Symmetric": symmetric_history, "Asymmetric": asymmetric_history}
histories = {"Asymmetric": asymmetric_history}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, hist in histories.items():
    epochs = [h["epoch"] for h in hist]
    axes[0].plot(epochs, [h["train_loss"] for h in hist], label=f"{name} train")
    axes[0].plot(epochs, [h["val_loss"] for h in hist], "--", label=f"{name} val")
    axes[1].plot(epochs, [h["train_acc"] for h in hist], label=f"{name} train")
    axes[1].plot(epochs, [h["val_vqa_acc"] for h in hist], "--", label=f"{name} val")

axes[0].set(xlabel="Epoch", ylabel="Loss", title="Loss")
axes[1].set(xlabel="Epoch", ylabel="Accuracy (%)", title="VQA Accuracy")
for ax in axes:
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.2 Comparison Bar Chart

In [ ]:
metric_names = ["val_vqa_acc"]
model_names = list(results.keys())
x = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(6, 5))
for i, name in enumerate(model_names):
    values = [results[name][m] for m in metric_names]
    bars = ax.bar(x + i * width, values, width, label=name)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{val:.1f}", ha="center", va="bottom", fontsize=10)

ax.set_xticks(x + width / 2)
ax.set_xticklabels([m.replace("_", " ").title() for m in metric_names])
ax.set_ylabel("Accuracy (%)")
ax.set_title("Model Comparison — VQA Accuracy")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "comparison_bar.png", dpi=150, bbox_inches="tight")
plt.show()

### Visualization PreparationRe-attach frozen encoders to trained fusion models and reload the raw datasetfor visualization. Training used precomputed features; visualization needs rawimages and question strings.

In [ ]:
# Re-instantiate the raw dataset for visualization (images + question strings)
raw_val_ds = VQADataset(
    questions_file=DATA_DIR / "questions" / "v2_OpenEnded_mscoco_val2014_questions.json",
    annotations_file=DATA_DIR / "answers" / "v2_mscoco_val2014_annotations.json",
    h5_path=h5_path,
    answer_to_idx=answer_to_idx,
    max_question_len=MAX_QUESTION_LEN,
    transform=get_image_transform("val"),
    max_samples=MAX_SAMPLES,
)

raw_val_loader = DataLoader(raw_val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)

if FREEZE_ENCODERS:
    class EndToEndVQAWrapper(nn.Module):
        """Wraps a trained fusion model with fresh frozen encoders for visualization."""

        def __init__(self, trained_fusion_model, embed_dim=EMBED_DIM):
            super().__init__()
            self.image_encoder = ImageEncoder(embed_dim, freeze=True)
            self.text_encoder = TextEncoder(embed_dim, freeze=True)
            self.trained_model = trained_fusion_model

        def forward(self, images, input_ids, attention_mask):
            img_feats = self.image_encoder(images)
            txt_feats = self.text_encoder(input_ids, attention_mask)
            return self.trained_model(img_feats, txt_feats, attention_mask)

    set_seed(SEED)
    e2e_asymmetric = EndToEndVQAWrapper(asymmetric_model).to(device)
    # e2e_symmetric = EndToEndVQAWrapper(symmetric_model).to(device)
    # e2e_models_dict = {"Asymmetric": e2e_asymmetric, "Symmetric": e2e_symmetric}
    e2e_models_dict = {"Asymmetric": e2e_asymmetric}
else:
    # Models are already end-to-end — use them directly
    # e2e_models_dict = {"Asymmetric": asymmetric_model, "Symmetric": symmetric_model}
    e2e_models_dict = {"Asymmetric": asymmetric_model}

print(f"raw_val_ds: {len(raw_val_ds):,} samples")
print("End-to-end models ready for visualization.")

### 5.3 Attention Heatmaps

Visualize what the asymmetric model attends to: which image regions light up for each question word, and which words are most important for each image patch.

In [ ]:
# Visualization helpers
MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])


def denormalize(img_tensor):
    """Convert normalised (3,H,W) tensor to (H,W,3) uint8 array."""
    img = img_tensor.cpu().numpy().transpose(1, 2, 0)
    img = img * STD + MEAN
    return np.clip(img * 255, 0, 255).astype(np.uint8)


def decode_tokens(input_ids):
    """Decode token IDs to readable strings."""
    tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")
    return [tokenizer.decode(tid) for tid in input_ids.tolist()]


@torch.no_grad()
def get_attention_weights(model, image, input_ids, attention_mask):
    """Run a single sample and return attention weight dicts."""
    model.eval()
    img = image.unsqueeze(0).to(device)
    ids = input_ids.unsqueeze(0).to(device)
    mask = attention_mask.unsqueeze(0).to(device)
    _, attn = model(img, ids, mask)
    return {k: v.cpu() for k, v in attn.items()}


def plot_image_attention(attn_t2i, image_tensor, tokens, question, top_tokens=4):
    """Overlay text->image attention heatmaps on the original image."""
    img_np = denormalize(image_tensor)
    attn = attn_t2i.squeeze(0).numpy()  # (N_txt, N_img)

    grid_size = int(np.sqrt(attn.shape[1] - 1))  # 24 for DINOv2 ViT-g/14 at 336x336 (14-px patches)
    attn_spatial = attn[:, 1:]  # drop CLS column

    # Count how many actual words exist (ignoring padding)
    real_token_count = len([t for t in tokens if t not in ['<pad>']])

    # Only average the attention maps of the real tokens
    combined = attn_spatial[:real_token_count].mean(axis=0).reshape(grid_size, grid_size)

    combined_resized = np.array(
        Image.fromarray(combined).resize(img_np.shape[:2][::-1], Image.BILINEAR))

    token_importance = attn_spatial.sum(axis=1)
    top_idx = token_importance.argsort()[-top_tokens:][::-1]

    n_cols = min(top_tokens, len(top_idx)) + 1
    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))
    if n_cols == 1:
        axes = [axes]

    axes[0].imshow(img_np)
    axes[0].imshow(combined_resized, alpha=0.5, cmap="jet")
    axes[0].set_title("Combined")
    axes[0].axis("off")

    for i, idx in enumerate(top_idx):
        if i + 1 >= len(axes):
            break
        token_attn = attn_spatial[idx].reshape(grid_size, grid_size)
        token_resized = np.array(
            Image.fromarray(token_attn).resize(img_np.shape[:2][::-1], Image.BILINEAR))
        label = tokens[idx] if tokens else f"token {idx}"
        axes[i + 1].imshow(img_np)
        axes[i + 1].imshow(token_resized, alpha=0.5, cmap="jet")
        axes[i + 1].set_title(f'"{label}"')
        axes[i + 1].axis("off")

    fig.suptitle(question, fontsize=12)
    fig.tight_layout()
    return fig


def plot_text_attention(attn_img_to_txt, tokens, question):
    """Bar chart of image->text attention per token."""
    attn = attn_img_to_txt.squeeze(0).numpy()
    token_weights = attn.mean(axis=0)

    fig, ax = plt.subplots(figsize=(6, max(3, len(tokens) * 0.35)))
    y_pos = np.arange(len(tokens))
    ax.barh(y_pos, token_weights, color="steelblue")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(tokens, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel("Mean attention weight")
    ax.set_title(f"Image -> Text attention\n{question}")
    fig.tight_layout()
    return fig

In [ ]:
# Generate attention maps for sample validation images
for i in range(min(5, len(raw_val_ds))):
    image, input_ids, attention_mask, answer_target = raw_val_ds[i]
    question = raw_val_ds.samples[i]["question"]
    tokens = decode_tokens(input_ids)

    attn = get_attention_weights(e2e_models_dict["Asymmetric"], image, input_ids, attention_mask)

    fig = plot_image_attention(attn["txt_to_img"], image, tokens, question)
    fig.savefig(FIGURES_DIR / f"attn_img_{i}.png", dpi=150, bbox_inches="tight")
    plt.show()

    fig = plot_text_attention(attn["img_to_txt"], tokens, question)
    fig.savefig(FIGURES_DIR / f"attn_txt_{i}.png", dpi=150, bbox_inches="tight")
    plt.show()

### 5.4 Qualitative Comparison Grid

Side-by-side predictions and attention maps for both models on the same samples.

In [ ]:
@torch.no_grad()
def qualitative_grid(models, dataset, idx_to_answer, n_samples=6, save_path=None):
    """Grid of qualitative examples comparing models."""
    sample_indices = torch.randperm(len(dataset))[:n_samples].tolist()
    model_names = list(models.keys())
    n_cols = 1 + len(model_names)
    n_rows = len(sample_indices)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    for row, idx in enumerate(sample_indices):
        image, input_ids, attention_mask, answer_target = dataset[idx]
        question = dataset.samples[idx]["question"]
        gt_answer = idx_to_answer[answer_target.argmax().item()]
        img_np = denormalize(image)

        # Column 0: original image + question
        axes[row, 0].imshow(img_np)
        axes[row, 0].set_title(f"Q: {question}\nGT: {gt_answer}", fontsize=9)
        axes[row, 0].axis("off")

        # Remaining columns: one per model
        for col, name in enumerate(model_names, start=1):
            model = models[name]
            attn = get_attention_weights(model, image, input_ids, attention_mask)

            img_t = image.unsqueeze(0).to(device)
            ids_t = input_ids.unsqueeze(0).to(device)
            mask_t = attention_mask.unsqueeze(0).to(device)
            logits, _ = model(img_t, ids_t, mask_t)
            pred_answer = idx_to_answer.get(logits.argmax(dim=1).item(), "???")

            # Attention heatmap
            attn_t2i = attn["txt_to_img"].squeeze(0).numpy()
            grid_size = int(np.sqrt(attn_t2i.shape[1] - 1))
            combined = attn_t2i[:, 1:].mean(axis=0).reshape(grid_size, grid_size)
            combined_resized = np.array(
                Image.fromarray(combined).resize(img_np.shape[:2][::-1], Image.BILINEAR))

            axes[row, col].imshow(img_np)
            axes[row, col].imshow(combined_resized, alpha=0.5, cmap="jet")
            marker = "correct" if pred_answer == gt_answer else "wrong"
            axes[row, col].set_title(f"{name}\nPred: {pred_answer} ({marker})", fontsize=9)
            axes[row, col].axis("off")

    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    return fig

In [ ]:
fig = qualitative_grid(
    e2e_models_dict, raw_val_ds, idx_to_answer, n_samples=6,
    save_path=str(FIGURES_DIR / "qualitative_grid.png"))
plt.show()

In [ ]:
# @torch.no_grad()
# def error_analysis(models, loader, idx_to_answer, skip_n=1):
#     print(f"Running Error Analysis... skipping {skip_n} cases")
#     asym_model = models["Asymmetric"]
#     sym_model = models["Symmetric"]
#
#     asym_model.eval()
#     sym_model.eval()
#
#     cases_found = 0
#
#     for images, input_ids, attention_mask, answers in loader:
#         images = images.to(device)
#         input_ids = input_ids.to(device)
#         attention_mask = attention_mask.to(device)
#         targets = answers.to(device).argmax(dim=1)
#
#         asym_logits, _ = asym_model(images, input_ids, attention_mask)
#         sym_logits, _ = sym_model(images, input_ids, attention_mask)
#
#         asym_preds = asym_logits.argmax(dim=1)
#         sym_preds = sym_logits.argmax(dim=1)
#
#         # Find condition: Asymmetric correct AND Symmetric wrong
#         mask = (asym_preds == targets) & (sym_preds != targets)
#         divergent_indices = mask.nonzero(as_tuple=True)[0]
#
#         for idx in divergent_indices:
#             if cases_found < skip_n:
#                 cases_found += 1
#                 continue
#
#             print("--- Found a divergent case ---")
#             print(f"Target Answer: {idx_to_answer[targets[idx].item()]}")
#             print(f"Asymmetric Guess: {idx_to_answer[asym_preds[idx].item()]} (Correct)")
#             print(f"Symmetric Guess: {idx_to_answer[sym_preds[idx].item()]} (Wrong)")
#
#             # Extract the specific sample for visualization
#             img = images[idx].cpu()
#             ids = input_ids[idx].cpu()
#             mask_ = attention_mask[idx].cpu()
#
#             # Decode tokens to reconstruct a readable question
#             tokens = decode_tokens(ids)
#             clean_tokens = [t.replace('\u0120', '') for t in tokens if t not in ['<pad>', '<s>', '</s>']]
#             question_str = " ".join(clean_tokens).strip() + "?"
#             print(f"Question: {question_str}")
#
#             # Get attention weights for BOTH models
#             attn_asym = get_attention_weights(asym_model, img, ids, mask_)
#             attn_sym = get_attention_weights(sym_model, img, ids, mask_)
#
#             print("\n=== ASYMMETRIC MODEL ATTENTION (Correct Guess) ===")
#             fig_img_asym = plot_image_attention(attn_asym["txt_to_img"], img, tokens, question_str)
#             plt.show()
#             fig_txt_asym = plot_text_attention(attn_asym["img_to_txt"], tokens, question_str)
#             plt.show()
#
#             print("\n=== SYMMETRIC MODEL ATTENTION (Wrong Guess) ===")
#             fig_img_sym = plot_image_attention(attn_sym["txt_to_img"], img, tokens, question_str)
#             plt.show()
#             fig_txt_sym = plot_text_attention(attn_sym["img_to_txt"], tokens, question_str)
#             plt.show()
#
#             return
#
# error_analysis(e2e_models_dict, raw_val_loader, idx_to_answer, skip_n=10)

# Task
Categorize the validation questions by type (e.g., 'Is/Are', 'How many', 'What color') and compute the accuracy for both the symmetric and asymmetric models per category, generating a grouped bar chart to visualize the results. Additionally, conduct a comprehensive modality ablation test by evaluating both models under three conditions: Full Data, Image-Blind (zeroed images), and Text-Blind (zeroed input IDs/masks), and generate a grouped bar chart showing the degradation. Finally, provide a brief summary of the insights gained from these new evaluation metrics to help frame the presentation.

## Question Type Accuracy Breakdown

### Subtask:
Categorize validation questions by type, calculate accuracy per category for both models, and plot a grouped bar chart.


In [ ]:
from collections import defaultdict

# 1. Categorize question function
def categorize_question(question):
    q_lower = question.lower().strip()
    if q_lower.startswith(('is ', 'are ', 'was ', 'were ', 'does ', 'do ', 'has ', 'have ', 'can ', 'could ', 'would ', 'should ')):
        return 'Yes/No'
    elif q_lower.startswith('how many'):
        return 'Count'
    elif q_lower.startswith('what color'):
        return 'Color'
    else:
        return 'Other'

@torch.no_grad()
def evaluate_by_question_type(models, loader, dataset):
    """Evaluate VQA accuracy broken down by question type.
    Uses loader for model inference, raw dataset for question text."""
    for model in models.values():
        model.eval()

    category_scores = {name: defaultdict(list) for name in models.keys()}
    global_idx = 0

    use_amp = USE_AMP and device.type == "cuda"

    for inp1, inp2, masks, answers in tqdm(loader, desc="Evaluating by question type"):
        inp1    = inp1.to(device)
        inp2    = inp2.to(device)
        masks   = masks.to(device)
        answers = answers.to(device)

        batch_size = inp1.size(0)

        preds_dict = {}
        for name, model in models.items():
            amp_ctx = torch.amp.autocast(device_type=device.type, dtype=AMP_DTYPE) if use_amp else nullcontext()
            with amp_ctx:
                logits, _ = model(inp1, inp2, masks)
            preds_dict[name] = logits.argmax(dim=1)

        # 3. Calculate category score per sample
        for i in range(batch_size):
            question = dataset.samples[global_idx + i]["question"]
            category = categorize_question(question)

            for name, preds in preds_dict.items():
                pred_idx = preds[i]
                pred_soft = answers[i, pred_idx]
                vqa_score = torch.clamp(pred_soft * 10.0 / 3.0, max=1.0).item()
                category_scores[name][category].append(vqa_score)

        global_idx += batch_size

    # 4. Aggregate scores
    category_acc = {name: {} for name in models.keys()}
    for name, cat_scores in category_scores.items():
        for cat, scores in cat_scores.items():
            category_acc[name][cat] = np.mean(scores) * 100

    return category_acc

# Use precomputed models + val_loader for speed, raw_val_ds for question text
category_acc = evaluate_by_question_type(
    # {"Symmetric": symmetric_model, "Asymmetric": asymmetric_model},
    {"Asymmetric": asymmetric_model},
    val_loader, raw_val_ds)

# 5. Generate a grouped bar chart
categories = ['Yes/No', 'Count', 'Color', 'Other']
# sym_acc = [category_acc["Symmetric"].get(cat, 0) for cat in categories]
asym_acc = [category_acc["Asymmetric"].get(cat, 0) for cat in categories]

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 6))
# bars1 = ax.bar(x - width/2, sym_acc, width, label='Symmetric', color='lightblue')
bars2 = ax.bar(x + width/2, asym_acc, width, label='Asymmetric', color='steelblue')

# for bars in [bars1, bars2]:
for bars in [bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=9)

ax.set_ylabel('Accuracy (%)')
ax.set_title('VQA Accuracy by Question Type')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES_DIR / "question_type_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

## Modality Ablation Test

Evaluate both models under three conditions: Full Data, Image-Blind (zeroed images), and Text-Blind (zeroed input IDs/masks), and generate a grouped bar chart showing the degradation.

In [ ]:
@torch.no_grad()
def evaluate_ablation(model, loader, condition):
    """Evaluate model under ablation conditions."""
    model.eval()
    vqa_acc_sum = 0.0
    total = 0
    use_amp = USE_AMP and device.type == "cuda"

    for inp1, inp2, masks, answers in tqdm(loader, desc=f"Eval {condition}"):
        if condition == "Image-Blind":
            inp1 = torch.zeros_like(inp1)
        elif condition == "Text-Blind":
            if FREEZE_ENCODERS:
                inp2 = torch.zeros_like(inp2)     # zero out precomputed text features
            else:
                inp2 = torch.ones_like(inp2)      # RoBERTa padding token ID = 1
            masks = torch.zeros_like(masks)       # zero attention mask in both cases

        inp1    = inp1.to(device)
        inp2    = inp2.to(device)
        masks   = masks.to(device)
        answers = answers.to(device)

        amp_ctx = torch.amp.autocast(device_type='cuda', dtype=AMP_DTYPE) if use_amp else nullcontext()
        with amp_ctx:
            logits, _ = model(inp1, inp2, masks)

        preds = logits.argmax(dim=1)
        pred_soft = answers[torch.arange(answers.size(0), device=answers.device), preds]
        vqa_scores = torch.clamp(pred_soft * 10.0 / 3.0, max=1.0)
        vqa_acc_sum += vqa_scores.sum().item()
        total += answers.size(0)

    return vqa_acc_sum / total * 100

# Use models + val_loader
# models_dict = {"Symmetric": symmetric_model, "Asymmetric": asymmetric_model}
models_dict = {"Asymmetric": asymmetric_model}

conditions = ["Full Data", "Image-Blind", "Text-Blind"]
# ablation_results = {"Symmetric": {}, "Asymmetric": {}}
ablation_results = {"Asymmetric": {}}

for model_name, model in models_dict.items():
    print(f"\nRunning ablation for {model_name}...")
    for cond in conditions:
        acc = evaluate_ablation(model, val_loader, cond)
        ablation_results[model_name][cond] = acc
        print(f"  {cond}: {acc:.2f}%")

# Plotting the results
x = np.arange(len(conditions))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 6))
# sym_accs = [ablation_results["Symmetric"][cond] for cond in conditions]
asym_accs = [ablation_results["Asymmetric"][cond] for cond in conditions]

# bars1 = ax.bar(x - width/2, sym_accs, width, label='Symmetric', color='lightblue')
bars2 = ax.bar(x + width/2, asym_accs, width, label='Asymmetric', color='steelblue')

# for bars in [bars1, bars2]:
for bars in [bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=9)

ax.set_ylabel('Accuracy (%)')
ax.set_title('Modality Ablation Test - Performance Degradation')
ax.set_xticks(x)
ax.set_xticklabels(conditions)
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES_DIR / "ablation_test.png", dpi=150, bbox_inches="tight")
plt.show()

# Task
Create a helper function `visualize_category_example` that searches the validation dataset for a question of a specified category and plots the attention heatmaps for both models. Use this function to find and visualize examples for the 'Yes/No', 'Count', 'Color', and 'Other' categories, adding appropriate text and code cells for each, and conclude with a brief summary of these visualizations.

## Define Category Visualization Helper

### Subtask:
Create a helper function to search for and visualize a specific question category.


In [ ]:
def visualize_category_example(category, dataset, models_dict, idx_to_answer):
    print(f"Searching for an example in category: {category}...")
    for i in range(len(dataset)):
        question = dataset.samples[i]["question"]
        if categorize_question(question) == category:
            image, input_ids, attention_mask, answer_target = dataset[i]
            gt_answer = idx_to_answer[answer_target.argmax().item()]

            img_t = image.unsqueeze(0).to(device)
            ids_t = input_ids.unsqueeze(0).to(device)
            mask_t = attention_mask.unsqueeze(0).to(device)

            # Decode tokens to reconstruct a readable question
            tokens = decode_tokens(input_ids)
            clean_tokens = [t.replace('Ġ', '') for t in tokens if t not in ['<pad>', '<s>', '</s>']]
            question_str = " ".join(clean_tokens).strip() + "?"

            print("\n" + "="*40)
            print(f"Category: {category}")
            print(f"Question: {question_str}")
            print(f"Target Answer: {gt_answer}")
            print("="*40)

            for model_name, model in models_dict.items():
                model.eval()
                with torch.no_grad():
                    logits, _ = model(img_t, ids_t, mask_t)
                pred_answer = idx_to_answer.get(logits.argmax(dim=1).item(), "???")
                print(f"{model_name} Guess: {pred_answer}")

            for model_name, model in models_dict.items():
                print(f"\n=== {model_name.upper()} MODEL ATTENTION ===")
                attn = get_attention_weights(model, image, input_ids, attention_mask)

                # Plot text -> image attention
                fig_img = plot_image_attention(attn["txt_to_img"], image, tokens, question_str)
                plt.show()

                # Plot image -> text attention
                fig_txt = plot_text_attention(attn["img_to_txt"], tokens, question_str)
                plt.show()

            return

    print(f"No example found for category: {category}")

## Visualize Yes/No Question

### Subtask:
Find and visualize an example for the 'Yes/No' question category.


In [ ]:
visualize_category_example("Yes/No", raw_val_ds, e2e_models_dict, idx_to_answer)

## Visualize Count Question

### Subtask:
Find and visualize an example for the 'Count' question category.

In [ ]:
visualize_category_example("Count", raw_val_ds, e2e_models_dict, idx_to_answer)

## Visualize Color Question

### Subtask:
Find and visualize an example for the 'Color' question category.

In [ ]:
visualize_category_example("Color", raw_val_ds, e2e_models_dict, idx_to_answer)

## Visualize Other Question

### Subtask:
Find and visualize an example for the 'Other' question category.

In [ ]:
visualize_category_example("Other", raw_val_ds, e2e_models_dict, idx_to_answer)

## Summary of Category Visualizations

Based on the attention heatmaps across different question categories ('Yes/No', 'Count', 'Color', 'Other'), we can observe the following:

1. **Asymmetric Model Focus**: The asymmetric model often demonstrates a more focused and interpretable attention mechanism. When asked about a specific object or its attribute (like 'color' or 'count'), the text-to-image attention effectively isolates the relevant regions of the image corresponding to the target words.
2. **Symmetric Model Limitations**: The symmetric model's attention maps tend to be more diffuse. Because it uses a single shared block for both directions, it struggles to decouple the distinct tasks of 'understanding the question' and 'locating the visual evidence'.
3. **Question-Specific Grounding**: In 'Count' and 'Color' questions, grounded visual evidence is crucial. The asymmetric model's ability to first process the text and then use it as a query to attend to the image allows it to better pinpoint the items to be counted or analyzed for color, leading to more accurate predictions.

In [ ]:
!mkdir -p /content/data/zip/
!mkdir -p /content/data/images/
!cp /content/drive/MyDrive/test2015.zip /content/data/zip/

In [ ]:
!unzip -q /content/data/zip/test2015.zip -d /content/data/images/

---
## 6. Test Set Predictions Export

Run inference on the VQA v2.0 **test2015** split (no annotations) and write predictions to JSON in the official VQA submission format: `[{"question_id": int, "answer": str}, ...]`.

Test images aren't in the precomputed feature HDF5, so we load raw JPEGs from `images/test2015/` and run them through the end-to-end models (`e2e_models_dict`, which works for both frozen and unfrozen runs).

In [ ]:
class VQATestDataset(Dataset):
    """VQA test split: questions + raw JPEGs from images/test2015/, no annotations."""

    def __init__(self, questions_file, images_dir, max_question_len=20,
                 transform=None, max_samples=None):
        self.images_dir = Path(images_dir) / "test2015"
        self.max_question_len = max_question_len
        self.transform = transform or get_image_transform("val")
        self.tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")

        with open(questions_file) as f:
            self.samples = json.load(f)["questions"]
        if max_samples is not None:
            self.samples = self.samples[:max_samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        # test-dev2015 reuses the test2015 image pool, so the filename prefix is
        # always COCO_test2015_* regardless of which question split we loaded.
        img_path = self.images_dir / f"COCO_test2015_{sample['image_id']:012d}.jpg"
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        encoding = self.tokenizer(
            sample["question"],
            max_length=self.max_question_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return (
            image,
            encoding["input_ids"].squeeze(0),
            encoding["attention_mask"].squeeze(0),
            sample["question_id"],
        )


# Use the full test2015 split (~447K questions) — required for any VQA v2
# eval-server submission (test-dev or test-standard). For a quick local
# sanity check during iteration, swap to v2_OpenEnded_mscoco_test-dev2015_questions.json (~107K).
TEST_QUESTIONS_FILE = DATA_DIR / "questions" / "v2_OpenEnded_mscoco_test2015_questions.json"

test_ds = VQATestDataset(
    questions_file=TEST_QUESTIONS_FILE,
    images_dir=DATA_DIR / "images",
    max_question_len=MAX_QUESTION_LEN,
    transform=get_image_transform("val"),
    max_samples=None,  # always export the full test split, regardless of dev MAX_SAMPLES
)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)
print(f"Test: {len(test_ds):,} samples ({len(test_loader)} batches)")


@torch.no_grad()
def predict_test(model, loader, idx_to_answer):
    """Return [{question_id, answer}, ...] in official VQA submission format."""
    model.eval()
    use_amp = USE_AMP and device.type == "cuda"
    predictions = []

    for images, input_ids, attention_mask, question_ids in tqdm(loader, desc="predict"):
        images = images.to(device)
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        amp_ctx = torch.amp.autocast(device_type=device.type, dtype=AMP_DTYPE) if use_amp else nullcontext()
        with amp_ctx:
            logits, _ = model(images, input_ids, attention_mask)

        preds = logits.argmax(dim=1).cpu().tolist()
        for qid, p in zip(question_ids.tolist(), preds):
            predictions.append({"question_id": int(qid), "answer": idx_to_answer[int(p)]})

    return predictions


PREDICTIONS_DIR = CHECKPOINT_DIR.parent / "predictions"
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

# for name, model in [("asymmetric", e2e_models_dict["Asymmetric"]),
#                     ("symmetric",  e2e_models_dict["Symmetric"])]:
for name, model in [("asymmetric", e2e_models_dict["Asymmetric"])]:
    preds = predict_test(model, test_loader, idx_to_answer)
    out_path = PREDICTIONS_DIR / f"{name}_test_predictions.json"
    with open(out_path, "w") as f:
        json.dump(preds, f)
    print(f"Saved {len(preds):,} {name} predictions -> {out_path}")


In [ ]:
!cp -r /content/results/* /content/drive/MyDrive/updated_unfrozen_results/